## 1. Configurações e carregamento do dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

dataset_file = '../data/500+.csv'
ano_predicao = 2026

def filtrar_inconsistencias(df_data):
    return df_data.loc[(df_data['Artista'] != '???') & (df_data['Musica'].str.len() > 0) & (df_data['Observacao'] != 'repetida')]

def load_data(agregar_pinkfloyd):
    df_data = pd.read_csv(dataset_file)
    df_data['Data_Lancamento_Album'] = pd.to_datetime(df_data['Data_Lancamento_Album'])
    df_data['Decada_Musica'] = (df_data['Data_Lancamento_Album'].dt.year // 10) * 10
    df_data['Ano_Musica'] = df_data['Data_Lancamento_Album'].dt.year
    df_data['Duracao'] = df_data.loc[:,'Duracao'].fillna(value=0)
    if (agregar_pinkfloyd):
        df_data.loc[df_data['Musica'].str.contains('Another Brick', na=False), 'Musica'] = 'Another Brick in the Wall'
        df_data.loc[df_data['Musica'].str.contains('Another Brick', na=False), 'Duracao'] = 508

    df_data = df_data.drop(['Artista_Origem', 'Musica_Origem', 'Artista_Wikidata_ID', 'Artista_Wikidata', 'Artista_Wiki', 'Country', 'Genre', 'Musica_Wikidata_ID', 'Musica_Wikidata', 'Musica_Wiki', 'Album_Single_Wikidata_ID', 'Album_Single_Wikidata', 'Album_Single_Wiki', 'Data_Lancamento_Album'], axis=1)
    df_data.rename(columns={'Album_Single':'Album'}, inplace=True)
    df_data = filtrar_inconsistencias(df_data)
    return df_data

df = load_data(True)
df = df[df['Ano'] < ano_predicao]
print("Dataset carregado com sucesso!")

## 2. Identificador única da música

In [ ]:
print("Preparando dados...")

# Criar identificador único: Artista-Musica-Observacao
df['id_musica'] = (df['Artista'].fillna('') + '|||' + 
                    df['Musica'].fillna('') + '|||' + 
                    df['Observacao'].fillna(''))

# Garantir que Observacao seja tratada corretamente
df['Observacao'] = df['Observacao'].fillna('')

print(f"\nTotal de músicas únicas: {df['id_musica'].nunique()}")
print(f"Total de registros: {len(df)}")

print(f"\nAno de previsão: {ano_predicao}")
print(f"Anos no dataset: {df['Ano'].min()} a {df['Ano'].max()}")

## 3. Engenharia de features

In [ ]:
def calcular_features_musica(df, ano_previsao):
    features_list = []
    
    # Obter todas as músicas únicas
    musicas_unicas = df['id_musica'].unique()
    
    for id_musica in musicas_unicas:
        df_musica = df[df['id_musica'] == id_musica].sort_values('Ano')
        
        # Informações básicas
        artista = df_musica['Artista'].iloc[0]
        musica = df_musica['Musica'].iloc[0]
        observacao = df_musica['Observacao'].iloc[0]
        pais = df_musica['Pais'].iloc[0] if 'Pais' in df_musica.columns else None
        genero = df_musica['Genero'].iloc[0] if 'Genero' in df_musica.columns else None
        
        # Anos de aparição
        anos_aparicao = df_musica['Ano'].values
        anos_totais = ano_previsao - df['Ano'].min()
        
        # 1. frequencia_aparicao: % de anos em que apareceu
        frequencia_aparicao = len(anos_aparicao) / anos_totais if anos_totais > 0 else 0
        
        # 2. streak_anos: Anos consecutivos (até o último ano)
        anos_ordenados = sorted(anos_aparicao, reverse=True)
        streak_anos = 0
        ano_esperado = ano_previsao - 1
        for ano in anos_ordenados:
            if ano == ano_esperado:
                streak_anos += 1
                ano_esperado -= 1
            else:
                break
        
        # 3. anos_desde_ultima: Anos desde última aparição
        ultimo_ano = max(anos_aparicao)
        anos_desde_ultima = ano_previsao - ultimo_ano - 1
        
        # 4. aparicao_unica: Flag binária
        aparicao_unica = 1 if len(anos_aparicao) == 1 else 0
        
        # 5. anos_desde_unica_aparicao
        anos_desde_unica_aparicao = anos_desde_ultima if aparicao_unica == 1 else 0
        
        # 6. dropout_score: Score de risco de dropout
        # Maior quando: aparição única antiga, ou muitos anos sem aparecer
        if aparicao_unica == 1:
            dropout_score = min(anos_desde_unica_aparicao / 10, 1.0)
        else:
            dropout_score = min(anos_desde_ultima / 5, 1.0) * (1 - frequencia_aparicao)
        
        # 7. forca_musica: Score composto de estabelecimento
        # Maior quanto mais frequente e recente
        forca_musica = (frequencia_aparicao * 0.5 + 
                       (1 - min(anos_desde_ultima / 10, 1.0)) * 0.3 +
                       min(streak_anos / 5, 1.0) * 0.2)
        
        # 8. penalidade_one_hit: Penalidade para one-hit wonders
        penalidade_one_hit = anos_desde_unica_aparicao * 0.1 if aparicao_unica == 1 else 0
        
        # 9. volatilidade_posicao: Amplitude entre melhor e pior posição
        posicoes = df_musica['Posicao'].values
        volatilidade_posicao = max(posicoes) - min(posicoes) if len(posicoes) > 1 else 0
        
        # 10. consistencia: Regularidade nas aparições
        if len(anos_aparicao) > 1:
            gaps = np.diff(sorted(anos_aparicao))
            consistencia = 1 / (1 + np.std(gaps)) if len(gaps) > 0 else 1
        else:
            consistencia = 0
        
        # Estatísticas de posição
        posicao_media = df_musica['Posicao'].mean()
        melhor_posicao = df_musica['Posicao'].min()
        pior_posicao = df_musica['Posicao'].max()
        ultima_posicao = df_musica[df_musica['Ano'] == ultimo_ano]['Posicao'].iloc[0]
        
        # Tendência de posição (melhorando ou piorando)
        if len(posicoes) > 1:
            tendencia_posicao = posicoes[-1] - posicoes[0]  # negativo = melhorando
        else:
            tendencia_posicao = 0
        
        features_list.append({
            'id_musica': id_musica,
            'Artista': artista,
            'Musica': musica,
            'Observacao': observacao,
            'Pais': pais,
            'Genero': genero,
            'frequencia_aparicao': frequencia_aparicao,
            'streak_anos': streak_anos,
            'anos_desde_ultima': anos_desde_ultima,
            'aparicao_unica': aparicao_unica,
            'anos_desde_unica_aparicao': anos_desde_unica_aparicao,
            'dropout_score': dropout_score,
            'forca_musica': forca_musica,
            'penalidade_one_hit': penalidade_one_hit,
            'volatilidade_posicao': volatilidade_posicao,
            'consistencia': consistencia,
            'posicao_media': posicao_media,
            'melhor_posicao': melhor_posicao,
            'pior_posicao': pior_posicao,
            'ultima_posicao': ultima_posicao,
            'tendencia_posicao': tendencia_posicao,
            'num_aparicoes': len(anos_aparicao),
            'ultimo_ano': ultimo_ano
        })
    
    return pd.DataFrame(features_list)

## 4. Preparação dos dados de treino

In [ ]:
print("Preparando dados de treino...")

anos = sorted(df['Ano'].unique())

dados_treino = []

for i in range(len(anos) - 1):
    ano_atual = anos[i]
    ano_proximo = anos[i + 1]
    
    # Features até o ano atual
    df_ate_ano = df[df['Ano'] <= ano_atual]
    features_ano = calcular_features_musica(df_ate_ano, ano_proximo)
    
    # Target: posição no próximo ano (ou 501 se não apareceu)
    df_proximo_ano = df[df['Ano'] == ano_proximo]
    
    for _, row in features_ano.iterrows():
        id_musica = row['id_musica']
        
        # Verificar se apareceu no próximo ano
        musica_proximo = df_proximo_ano[df_proximo_ano['id_musica'] == id_musica]
        
        if len(musica_proximo) > 0:
            posicao_proxima = musica_proximo['Posicao'].iloc[0]
            apareceu = 1
        else:
            posicao_proxima = 501  # Não apareceu
            apareceu = 0
        
        dados_treino.append({
            **row.to_dict(),
            'ano_previsao': ano_proximo,
            'posicao_proxima': posicao_proxima,
            'apareceu_proximo': apareceu
        })
    
    df_treino = pd.DataFrame(dados_treino)

    print(f"Exemplos de treino: {len(df_treino)}")

## 5. Treinamento do modelo

In [ ]:
print("Treinando modelos...")

# Features para o modelo
feature_cols = [
    'frequencia_aparicao', 'streak_anos', 'anos_desde_ultima',
    'aparicao_unica', 'anos_desde_unica_aparicao', 'dropout_score',
    'forca_musica', 'penalidade_one_hit', 'volatilidade_posicao',
    'consistencia', 'posicao_media', 'melhor_posicao', 
    'ultima_posicao', 'tendencia_posicao', 'num_aparicoes'
]

X = df_treino[feature_cols].fillna(0)

# Modelo 1: Prever se vai aparecer (classificação binária)
y_aparece = df_treino['apareceu_proximo']

# Modelo 2: Prever posição (apenas para músicas que aparecem)
df_apareceu = df_treino[df_treino['apareceu_proximo'] == 1]
X_pos = df_apareceu[feature_cols].fillna(0)
y_pos = df_apareceu['posicao_proxima']

# Treinar modelos
print("Treinando modelo de aparição...")
modelo_aparicao = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
modelo_aparicao.fit(X, y_aparece)

print("Treinando modelo de posição...")
modelo_posicao = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
modelo_posicao.fit(X_pos, y_pos)

# Importância das features
print("\nImportância das Features (Aparição):")
importancias = pd.DataFrame({
    'feature': feature_cols,
    'importancia': modelo_aparicao.feature_importances_
}).sort_values('importancia', ascending=False)
print(importancias.head(10))

## 6. Geração do ranking

In [ ]:
# =============================================================================
# 6. GERAÇÃO DO RANKING PREVISTO
# =============================================================================
print(f"Gerando ranking previsto para {ano_predicao}...")

# Calcular features para todas as músicas
features_atual = calcular_features_musica(df, ano_predicao)

X = features_atual[feature_cols].fillna(0)

# Prever probabilidade de aparecer
prob_aparecer = modelo_aparicao.predict(X)
prob_aparecer = np.clip(prob_aparecer, 0, 1)

# Prever posição
posicao_prevista = modelo_posicao.predict(X)
posicao_prevista = np.clip(posicao_prevista, 1, 500)

# Combinar resultados
features_atual['prob_aparecer'] = prob_aparecer
features_atual['posicao_prevista'] = posicao_prevista

# Score combinado: músicas com maior probabilidade e melhor posição
features_atual['score_final'] = (
    features_atual['prob_aparecer'] * 100 - 
    features_atual['posicao_prevista'] * 0.1
)

# Ordenar por score
ranking = features_atual.sort_values('score_final', ascending=False).copy()

# Adicionar posição no ranking
ranking['posicao_ranking'] = range(1, len(ranking) + 1)

# Normalizar probabilidades para que somem 100% em cada posição
# (probabilidade de estar exatamente naquela posição)
ranking['prob_posicao_exata'] = ranking['prob_aparecer'] / ranking['prob_aparecer'].sum()
ranking['prob_posicao_exata'] = ranking['prob_posicao_exata'] * 100

print("\n" + "=" * 80)
print(f"RANKING PREVISTO PARA {ano_predicao}")
print("=" * 80)
print("\nTop 20 músicas:")
print(ranking.head(20).to_string(index=False))

## 7. Visualização

In [ ]:
print("\n" + "=" * 80)
print("ANÁLISE DOS RESULTADOS")
print("=" * 80)

print(f"\nTotal de músicas no ranking: {len(ranking)}")
print(f"\nProbabilidade média de aparecer: {ranking['prob_aparecer'].mean():.2%}")
print(f"Força média das músicas: {ranking['forca_musica'].mean():.3f}")
print(f"Dropout score médio: {ranking['dropout_score'].mean():.3f}")

print("\nDistribuição de Streak Anos:")
print(ranking['streak_anos'].value_counts().sort_index().head(10))

print("\nMúsicas com maior probabilidade de aparecer:")
print(ranking.nlargest(10, 'prob_aparecer')[
    ['posicao_ranking', 'Artista', 'Musica', 'prob_aparecer', 'forca_musica']
].to_string(index=False))

print("\nMúsicas em risco (maior dropout_score):")
print(ranking.nlargest(10, 'dropout_score')[
    ['posicao_ranking', 'Artista', 'Musica', 'dropout_score', 'streak_anos']
].to_string(index=False))

## 8. Exportação

In [ ]:
ranking = ranking[['posicao_ranking', 'Artista', 'Musica', 'Observacao', 
                'prob_aparecer', 'prob_posicao_exata', 'posicao_prevista',
                'forca_musica', 'dropout_score', 'streak_anos']]
ranking.to_csv("../data/predicao_proximo_ano.csv", index=False)